# Deep Learning Practicals
**Subject:** 410251 - Deep Learning  
**Practicals covered:**
1. Linear Regression using Deep Neural Network — Boston Housing
2. Binary Classification using DNN — IMDB Sentiment
3. Convolutional Neural Network — Fashion MNIST
4. Recurrent Neural Network — Google Stock Price Prediction

## DL-1: Linear Regression using Deep Neural Network
**Problem:** Implement Boston housing price prediction using Linear regression with Deep Neural network.  
**Dataset:** Boston House price prediction dataset.

In [ ]:
# Boston Housing Price Prediction
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import sys
print("Libraries imported successfully.")

In [ ]:
# Load the data
print("Fetching Boston Housing dataset from OpenML...")
boston = fetch_openml(name='boston', version=1, as_frame=True, parser='auto')
X, y = boston.data, boston.target.astype(np.float32)

# Ensure all features are numeric and handle missing values
X = X.apply(pd.to_numeric, errors='coerce')
if X.isnull().values.any():
    X = X.fillna(X.median())
    print("Handled missing values.")

X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
feature_scaler = StandardScaler()
x_train_scaled = feature_scaler.fit_transform(X_train_all)
x_test_scaled = feature_scaler.transform(X_test)
y_train = y_train_all.to_numpy(dtype=np.float32).reshape(-1, 1)
y_test = y_test.to_numpy(dtype=np.float32).reshape(-1, 1)
print(f"Training samples: {x_train_scaled.shape[0]}")
print(f"Test samples: {x_test_scaled.shape[0]}")

In [ ]:
# Define the Model
class BostonRegressionNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BostonRegressionNet(x_train_scaled.shape[1]).to(device)
print(f"Model initialized on {device}.")

In [ ]:
# Training setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
tensor_x_train = torch.tensor(x_train_scaled, dtype=torch.float32).to(device)
tensor_y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
train_dataset = TensorDataset(tensor_x_train, tensor_y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

num_epochs = 200
best_loss = float('inf')
patience = 15
epochs_no_improve = 0
best_state = None

print("Starting training...")
for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_losses = []
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    epoch_loss = np.mean(epoch_losses)

    if epoch % 20 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | Train MSE: {epoch_loss:.4f}')
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        epochs_no_improve = 0
        best_state = model.state_dict()
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping triggered at epoch {epoch}')
            break

if best_state is not None:
    model.load_state_dict(best_state)
print("Training complete.")

In [ ]:
# Evaluation
model.eval()
tensor_x_test = torch.tensor(x_test_scaled, dtype=torch.float32).to(device)
with torch.no_grad():
    y_pred = model(tensor_x_test).cpu().numpy().flatten()

y_test_flat = y_test.flatten()
mse = mean_squared_error(y_test_flat, y_pred)
mae = mean_absolute_error(y_test_flat, y_pred)
r2 = r2_score(y_test_flat, y_pred)

print('\nBoston Housing Price Prediction Results')
print('--------------------------------------')
print(f'Test MSE : {mse:.3f}')
print(f'Test MAE : {mae:.3f}')
print(f'Test R2  : {r2:.3f}')
print('\nSample predictions (Actual vs Predicted):')
for actual, predicted in list(zip(y_test_flat[:10], y_pred[:10])):
    print(f'Actual: {actual:.1f}, Predicted: {predicted:.1f}')

## DL-2: Binary Classification using Deep Neural Networks
**Problem:** Classify movie reviews into positive/negative based on text content.  
**Dataset:** IMDB dataset.

In [ ]:
# IMDB Movie Review Sentiment Classification
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
import matplotlib.pyplot as plt
print("TensorFlow version:", tf.__version__)

In [ ]:
# Load the dataset
vocab_size = 10000
max_length = 250
print("Loading IMDB dataset...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# Pad sequences to ensure uniform input size
x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)
print(f"Training reviews: {len(x_train)}")
print(f"Test reviews: {len(x_test)}")

In [ ]:
# Define the model architecture
embedding_dim = 16

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Train the model
epochs = 10
history = model.fit(
    x_train, y_train,
    epochs=epochs,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Visualize training results
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.show()

In [ ]:
# Helper function to test your own reviews
word_index = imdb.get_word_index()

def predict_sentiment(review_text):
    # Preprocess the text to match model training
    words = review_text.lower().split()
    review_seq = [word_index.get(w, 0) + 3 for w in words]
    review_seq = [i if i < vocab_size else 0 for i in review_seq]
    padded = pad_sequences([review_seq], maxlen=max_length)

    prediction = model.predict(padded)[0][0]
    sentiment = "POSITIVE" if prediction > 0.5 else "NEGATIVE"
    return sentiment, prediction

# Example
my_review = "This movie was absolutely wonderful. The acting was great and the plot was gripping."
sentiment, score = predict_sentiment(my_review)
print(f"Review: {my_review}")
print(f"Sentiment: {sentiment} (Score: {score:.4f})")

## DL-3: Convolutional Neural Network (CNN)
**Problem:** Create a classifier to classify fashion clothing into categories.  
**Dataset:** MNIST Fashion Dataset.

In [ ]:
# Fashion MNIST Classification using CNN
import numpy as np
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
print("TensorFlow version:", tf.__version__)

In [ ]:
# Load and preprocess the dataset
print("Loading Fashion MNIST dataset...")
(train_images, train_labels), (test_images, test_labels) = datasets.fashion_mnist.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

# Reshape images to include channel dimension (28, 28, 1)
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training set: {train_images.shape}")
print(f"Test set: {test_images.shape}")

In [ ]:
# Visualize some samples
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i].reshape(28, 28), cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])
plt.show()

In [ ]:
# Build the CNN architecture
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Train the model
history = model.fit(train_images, train_labels, epochs=10,
                    validation_data=(test_images, test_labels))

In [ ]:
# Plot accuracy and loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.show()

In [ ]:
# Make predictions and visualize results
predictions = model.predict(test_images)

def plot_image(i, predictions_array, true_label, img):
    true_label, img = true_label[i], img[i].reshape(28, 28)
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(img, cmap=plt.cm.binary)
    predicted_label = np.argmax(predictions_array)
    color = 'blue' if predicted_label == true_label else 'red'
    plt.xlabel(f"{class_names[predicted_label]} {100*np.max(predictions_array):2.0f}% ({class_names[true_label]})", color=color)

num_rows = 5
num_cols = 3
num_images = num_rows * num_cols
plt.figure(figsize=(2 * 2 * num_cols, 2 * num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2 * num_cols, 2 * i + 1)
    plot_image(i, predictions[i], test_labels, test_images)
plt.show()

## DL-4: Recurrent Neural Network (RNN)
**Problem:** Design a time series analysis and prediction system using RNN.  
**Dataset:** Google stock prices dataset.

In [ ]:
# Step 1: Install Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from sklearn.preprocessing import MinMaxScaler
import yfinance as yf
print("Libraries imported successfully. TensorFlow version:", tf.__version__)

In [ ]:
# Fetch Google (GOOGL) Stock Data
print("Downloading Google stock data...")
data = yf.download('GOOGL', start='2010-01-01', end='2023-12-31')

# Preprocess for training
training_set = data.iloc[:, 0:1].values

plt.figure(figsize=(14, 5))
plt.plot(data['Open'])
plt.title('Google Stock Price History')
plt.show()

In [ ]:
# Feature Scaling
sc = MinMaxScaler(feature_range=(0, 1))
training_set_scaled = sc.fit_transform(training_set)

# Create sliding window (60 days)
X_train = []
y_train = []
for i in range(60, len(training_set_scaled)):
    X_train.append(training_set_scaled[i-60:i, 0])
    y_train.append(training_set_scaled[i, 0])
X_train, y_train = np.array(X_train), np.array(y_train)

# Reshape for LSTM [samples, timesteps, features]
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
print(f"Data preprocessed. X_train shape: {X_train.shape}")

In [ ]:
# Build the RNN model
model = Sequential([
    LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    LSTM(units=50, return_sequences=True),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(units=1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
print("Model summary:")
model.summary()

In [ ]:
print("Training the RNN (20 epochs)... This may take a moment.")
model.fit(X_train, y_train, epochs=20, batch_size=32)

In [ ]:
predicted_stock_price = model.predict(X_train)
predicted_stock_price = sc.inverse_transform(predicted_stock_price)

plt.figure(figsize=(14, 5))
plt.plot(training_set[60:], color='red', label='Real Price')
plt.plot(predicted_stock_price, color='blue', label='Predicted Price')
plt.title('Google Stock Price Prediction Result')
plt.legend()
plt.show()